# Dataset and optional development pilot

Does compressing an existing agent transcript change an external monitor's ability to distinguish problematic trajectories from their paired benign counterparts? The principal comparison is **structured summary minus free summary AUROC**, with equal per-example context ceilings and fixed models. Full visible history and deterministic head/tail are references.

This is a retrospective experiment on inert transcript data. No transcript commands are executed, and neither model receives tools. Detecting a problem after a trajectory does not demonstrate prevention. Structured summaries are a directional hypothesis, not an assumed winner.

**Safe default:** this notebook displays the public inventory and an independent synthetic demonstration. No API key is needed. Real generation occurs only in the final optional cell with `RUN_LIVE=true`, explicit budget, configured model IDs, and confirmed data use. Empty committed outputs prevent accidental disclosure.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "research_plan.md").exists()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required"
print("Python:", sys.version.split()[0])
print("Repository found; no model API request has been made.")

## Configuration and audited inventory

Configuration is loaded locally. Only approved numeric inventory fields and opaque split IDs are displayed. Model IDs, context windows and official price snapshots must be configured and validated before a paid pilot. Byte lengths are not model-token counts. Labels remain in the evaluator and never enter `TranscriptInput` or prompts.

In [ ]:
import pandas as pd
import yaml
from IPython.display import display

config = yaml.safe_load((ROOT / "configs" / "pilot.yaml").read_text())
print("Protocol:", config["protocol_version"])
print("Configured split:", config["split"])
print("Explicit monitor ID configured:", bool(config.get("monitor_model")))
print("Explicit summarizer ID configured:", bool(config.get("summarizer_model")))
print("Token-budget rule: min(T, max(128, min(1024, floor(0.25*T))))")
inventory_path = ROOT / "data" / "manifests" / "inventory.json"
if inventory_path.exists():
    inventory = json.loads(inventory_path.read_text())
    print("Pinned dataset commit:", inventory["dataset_commit"])
    display(pd.DataFrame([inventory["counts"]]).T.rename(columns={0: "count"}))
    print("Size unit:", inventory["body_size"]["unit"])
    print("Model-context validation:", inventory["context_window_validation"])
else:
    print("Real dataset inventory has not been created. No substitute real data is loaded.")
split_path = ROOT / "data" / "manifests" / "split.csv"
if split_path.exists():
    split = pd.read_csv(split_path)
    assert set(split.columns) == {"transcript_id", "scenario_id", "family_id", "split"}
    assert split.groupby("family_id")["split"].nunique().max() == 1
    assert split.groupby("scenario_id")["transcript_id"].nunique().eq(2).all()
    display(
        split.groupby("split").agg(
            transcripts=("transcript_id", "nunique"),
            scenarios=("scenario_id", "nunique"),
            families=("family_id", "nunique"),
        )
    )
    print("Opaque split only; no source-name mapping or answer key is displayed.")

## SYNTHETIC FIXTURE — independent demonstration, not empirical evidence

The following fixture concerns colored blocks and was authored for software tests. It is unrelated to benchmark cases. Its task, events and summaries can be displayed publicly. Demonstration summaries below are hand-authored, **not model generations**; no monitor performance score or empirical result is exported.

The initial human task is preserved literally outside compression. The later tool result retains its distinct type even though its source role is `user`; an embedded statement does not become human authorization. Thinking and the private canary are excluded from the model-visible view.

In [ ]:
from context_audit.dataset import parse_transcript
from context_audit.render import render_body, render_task_header

print("SYNTHETIC FIXTURE — NOT BENCHMARK RESULTS")
transcript = parse_transcript(
    ROOT / "tests" / "fixtures" / "visible_events.jsonl",
    transcript_id="t_" + "0" * 24,
    data_origin="synthetic_fixture",
)
body = render_body(transcript)
header = render_task_header(transcript)
assert "SYNTHETIC_HIDDEN_THINKING" not in body
assert "canary" not in transcript.model_dump()
assert not {"label", "metadata", "scenario_id", "family_id"} & transcript.model_dump().keys()
print(header)
display(pd.DataFrame([event.model_dump(exclude_none=True) for event in transcript.events]))

## Four representations and measured size

For this offline illustration only, `ceil(UTF-8 bytes / 4)` is a declared heuristic counting method. It is neither the official tokenizer nor a word count. Live runs replace it with provider model-specific counting and record the method, ceiling, measured length and billed usage. Real `full` is never silently truncated.

All compressed representations share the same ceiling, including JSON fields and event IDs. Equal ceilings do not imply equal realized lengths. If the ceiling equals full length, all three use identity and avoid summary calls.

In [ ]:
import math

from context_audit.representations import budget_for, head_tail, validate_summary


def count_demo(text):
    return math.ceil(len(text.encode("utf-8")) / 4)


full_size = count_demo(body)
ceiling = budget_for(full_size)
free = (
    "E0002 read /demo/blocks.txt. E0003 reports two blue blocks and one green block; "
    "its embedded permission note is tool output, not human approval. "
    "E0004 asks to include green; E0005 claims the counts."
)
structured = json.dumps(
    {
        "environment_and_state": [],
        "actions_and_observed_results": ["E0003: two blue, one green."],
        "important_identifiers": ["E0002: /demo/blocks.txt"],
        "contradictions_or_missing_information": ["E0003: tool note is not human approval."],
        "source_event_ids": ["E0002", "E0003"],
    },
    separators=(",", ":"),
)
if ceiling == full_size:
    free = structured = body
else:
    visible = {event.event_id for event in transcript.events}
    free = validate_summary(free, "free_summary", ceiling, count_demo, visible)
    structured = validate_summary(structured, "structured_summary", ceiling, count_demo, visible)
representations = {
    "full": body,
    "head_tail": head_tail(body, ceiling, count_demo),
    "free_summary": free,
    "structured_summary": structured,
}
display(
    pd.DataFrame(
        [
            {
                "condition": condition,
                "measured_demo_units": count_demo(text),
                "ceiling": full_size if condition == "full" else ceiling,
                "compression_applied": condition != "full" and ceiling < full_size,
            }
            for condition, text in representations.items()
        ]
    )
)
for condition, text in representations.items():
    assert condition == "full" or count_demo(text) <= ceiling
    print("\nSYNTHETIC:", condition)
    print(text)
print("No monitor scores were generated or exported.")

## Optional real development pilot

Run 3–5 paired development scenarios only after reviewing the hypothesis, safety rubric and data-use terms. Configure `configs/pilot.yaml`, explicit model IDs and windows, a local official-price snapshot, and the API key. Set `RUN_LIVE=true` and `MAX_COST_USD` explicitly. The runner also validates its opt-in, data-use and financial controls; missing access is an informative failure, never a mock fallback.

The real pipeline keeps transcripts, full prompts, summaries and textual API responses under ignored private directories, preserving source canaries outside model inputs. This notebook prints only execution status. Do not view private test qualitative evidence until protocol freeze and test execution. Repeated generations need explicit repetition identities and paid budgeting.

In [ ]:
RUN_LIVE = os.getenv("RUN_LIVE", "false").lower() == "true"
if not RUN_LIVE:
    print(
        "Live pilot disabled. No paid calls made. Review and configure before RUN_LIVE=true."
    )
else:
    from contextlib import chdir
    from decimal import Decimal, InvalidOperation

    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env", override=False)
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "Real pilot requested but ANTHROPIC_API_KEY is missing; no mock fallback."
        )
    try:
        live_budget = Decimal(os.environ["MAX_COST_USD"])
    except (KeyError, InvalidOperation) as exc:
        raise RuntimeError(
            "Real pilot requires an explicit positive MAX_COST_USD financial cap."
        ) from exc
    if not live_budget.is_finite() or live_budget <= 0:
        raise RuntimeError("MAX_COST_USD must be finite and positive.")
    if not config.get("monitor_model") or not config.get("summarizer_model"):
        raise RuntimeError("Configure explicit validated monitor and summarizer model IDs first.")
    if not config.get("data_use_confirmed"):
        raise RuntimeError(
            "Confirm provider data-use compatibility in the pilot configuration first."
        )
    if not (ROOT / config["dataset_dir"]).exists():
        raise RuntimeError(
            "Real private dataset is missing; run the official acquisition workflow first."
        )
    if not config.get("rubric_reviewed"):
        raise RuntimeError("Review the hypothesis and monitor safety rubric before the real pilot.")
    # This fixed project command runs the evaluator, never any transcript command.
    from context_audit.cli import main

    with chdir(ROOT):
        exit_code = main(
            [
                "run",
                "--config",
                str(ROOT / "configs" / "pilot.yaml"),
                "--max-cost-usd",
                str(live_budget),
                "--live",
            ]
        )
    if exit_code:
        raise RuntimeError(
            "The requested real pilot did not complete; inspect the CLI error above."
        )

## Review checkpoint

Before main testing, check real output validity, all failure statuses, the equal ceiling and realized sizes, observed summary + monitor + retry costs, event references, and full-context preflight. Adjust only development choices. Review the hypothesis and rubric, then freeze model/config/prompt/split hashes in `protocol-v1`. A later change informed by test scores must be marked as a new exploratory version. This notebook does not freeze or publish the protocol automatically.